# Baixar população dos municípios do Ceará - IBGE

Este notebook baixa a população dos municípios do Ceará no Censo 2022 e salva em `api/populacao_ceara_ibge_2022.json` para o `app.py` usar localmente.

In [3]:
from pathlib import Path
import json
import unicodedata
import requests
import pandas as pd

PASTA_API = Path('../data/ibge_csv')
PASTA_API.mkdir(exist_ok=True)

ARQUIVO_SAIDA_JSON = PASTA_API / 'populacao_ceara_ibge_2022.json'
ARQUIVO_SAIDA_CSV = PASTA_API / 'populacao_ceara_ibge_2022.csv'

def normalizar_nome(nome):
    nome = str(nome).strip().upper()
    nome = unicodedata.normalize('NFKD', nome)
    nome = ''.join(c for c in nome if not unicodedata.combining(c))
    return nome


In [4]:
# API de Agregados do IBGE
# Tabela 4714: População residente, área territorial e densidade demográfica
# Variável 93: População residente
# Período 2022: Censo 2022
# Localidades: municípios da UF Ceará, código N3[23]

url = (
    'https://servicodados.ibge.gov.br/api/v3/agregados/4714/'
    'periodos/2022/variaveis/93'
    '?localidades=N6[N3[23]]'
)

resposta = requests.get(url, timeout=60)
print('Status:', resposta.status_code)
resposta.raise_for_status()

dados = resposta.json()
dados[:1]


Status: 200


[{'id': '93',
  'variavel': 'População residente',
  'unidade': 'Pessoas',
  'resultados': [{'classificacoes': [],
    'series': [{'localidade': {'id': '2300101',
       'nivel': {'id': 'N6', 'nome': 'Município'},
       'nome': 'Abaiara - CE'},
      'serie': {'2022': '10038'}},
     {'localidade': {'id': '2300150',
       'nivel': {'id': 'N6', 'nome': 'Município'},
       'nome': 'Acarape - CE'},
      'serie': {'2022': '14027'}},
     {'localidade': {'id': '2300200',
       'nivel': {'id': 'N6', 'nome': 'Município'},
       'nome': 'Acaraú - CE'},
      'serie': {'2022': '65264'}},
     {'localidade': {'id': '2300309',
       'nivel': {'id': 'N6', 'nome': 'Município'},
       'nome': 'Acopiara - CE'},
      'serie': {'2022': '44962'}},
     {'localidade': {'id': '2300408',
       'nivel': {'id': 'N6', 'nome': 'Município'},
       'nome': 'Aiuaba - CE'},
      'serie': {'2022': '14076'}},
     {'localidade': {'id': '2300507',
       'nivel': {'id': 'N6', 'nome': 'Município'},
       

In [5]:
registros = []

for variavel in dados:
    for resultado in variavel.get('resultados', []):
        for serie in resultado.get('series', []):
            localidade = serie.get('localidade', {})
            municipio = localidade.get('nome')
            codigo_ibge = localidade.get('id')
            valor = serie.get('serie', {}).get('2022')

            if not municipio or valor in [None, '', '-', '...', '..']:
                continue

            registros.append({
                'codigo_ibge': str(codigo_ibge),
                'municipio': municipio,
                'municipio_normalizado': normalizar_nome(municipio),
                'populacao': int(float(valor))
            })

df_pop = pd.DataFrame(registros).sort_values('municipio').reset_index(drop=True)
print(df_pop.shape)
df_pop.head()


(184, 4)


,codigo_ibge,municipio,municipio_normalizado,populacao
0,2300101,Abaiara - CE,ABAIARA - CE,10038
1,2300150,Acarape - CE,ACARAPE - CE,14027
2,2300200,Acaraú - CE,ACARAU - CE,65264
3,2300309,Acopiara - CE,ACOPIARA - CE,44962
4,2300408,Aiuaba - CE,AIUABA - CE,14076


In [6]:
populacao_por_nome = {
    linha['municipio_normalizado']: {
        'codigo_ibge': linha['codigo_ibge'],
        'municipio': linha['municipio'],
        'populacao': int(linha['populacao'])
    }
    for _, linha in df_pop.iterrows()
}

with open(ARQUIVO_SAIDA_JSON, 'w', encoding='utf-8') as f:
    json.dump(populacao_por_nome, f, ensure_ascii=False, indent=2)

df_pop.to_csv(ARQUIVO_SAIDA_CSV, index=False, encoding='utf-8-sig')

print('Arquivos salvos:')
print(ARQUIVO_SAIDA_JSON)
print(ARQUIVO_SAIDA_CSV)


Arquivos salvos:
..\data\ibge_csv\populacao_ceara_ibge_2022.json
..\data\ibge_csv\populacao_ceara_ibge_2022.csv
